# #69 2016~2024 총세출 대비 계획예산 비율 패널 구축

## tl;dr

- #62 재정대응 패널 153행에 지역통합 3개 회계 당초예산 순계를 1:1 결합한다.
- 명목 계획예산과 명목 총세출을 같은 백만원 단위로 나누어 주 비율을 산출한다.
- 5개 분모 대안별 비율 765행을 별도로 보존한다.

## Context & Methods

### Key Assumptions

- 계획예산 비율은 실질화 금액이 아닌 같은 연도의 명목 금액끼리 계산한다.
- 주 분모는 지역통합 3개 회계 당초예산 순계다.
- 100% 초과는 삭제하지 않고 분자·분모 행정범위 불일치 신호로 표시한다.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ANALYSIS_DIR = ROOT / "data/processed/analysis"
RESPONSE_PATH = ANALYSIS_DIR / "2016-2024_재정대응지수_패널.csv"
DENOMINATOR_PATH = ANALYSIS_DIR / "2016-2024_시도별_총세출_분모_패널.csv"
SENSITIVITY_PATH = ANALYSIS_DIR / "2016-2024_시도별_총세출_분모_민감도_패널.csv"
OUTPUT_PATH = ANALYSIS_DIR / "2016-2024_재정대응지수_총세출결합_패널.csv"
SENSITIVITY_OUTPUT_PATH = ANALYSIS_DIR / "2016-2024_재정대응지수_총세출_민감도_패널.csv"
KEY = ["지역", "연도"]

## Data

In [2]:
# 1. 입력 자료와 식별정보 로드
response = pd.read_csv(RESPONSE_PATH, encoding="utf-8-sig")
denominator = pd.read_csv(DENOMINATOR_PATH, encoding="utf-8-sig")
denominator_sensitivity = pd.read_csv(SENSITIVITY_PATH, encoding="utf-8-sig")

manifest = pd.DataFrame(
    [
        {
            "파일": source.name,
            "행": len(frame),
            "sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
        }
        for source, frame in [
            (RESPONSE_PATH, response),
            (DENOMINATOR_PATH, denominator),
            (SENSITIVITY_PATH, denominator_sensitivity),
        ]
    ]
)
assert response.duplicated(KEY).sum() == 0
assert denominator.duplicated(KEY).sum() == 0
assert denominator_sensitivity.duplicated(["분모대안", *KEY]).sum() == 0
display(manifest[["파일", "행"]])

,파일,행
0,2016-2024_재정대응지수_패널.csv,153
1,2016-2024_시도별_총세출_분모_패널.csv,153
2,2016-2024_시도별_총세출_분모_민감도_패널.csv,765


## Results

In [3]:
# 2. 주 분모 1:1 결합과 비율 산출
main_columns = KEY + [
    "예산단계",
    "포함회계",
    "자치단체수",
    "세출예산순계액_원",
    "세출예산순계액_백만원",
    "출처",
]
combined = response.merge(
    denominator[main_columns],
    on=KEY,
    how="left",
    validate="one_to_one",
    indicator=True,
)
combined["계획예산_총세출비율_pct"] = (
    combined["당해계획예산_백만원"] / combined["세출예산순계액_백만원"] * 100
)
combined["분모대안"] = "지역통합_3개회계_순계"
combined["분자분모_단위"] = "백만원"
display(
    combined[
        KEY + ["당해계획예산_백만원", "세출예산순계액_백만원", "계획예산_총세출비율_pct"]
    ].head()
)

,지역,연도,당해계획예산_백만원,세출예산순계액_백만원,계획예산_총세출비율_pct
0,강원,2016,966330.0,9.415973e+06,10.262667
1,강원,2017,1189656.0,9.521901e+06,12.493892
2,강원,2018,1070709.0,1.015499e+07,10.543671
3,강원,2019,1610182.0,1.114637e+07,14.445796
4,강원,2020,1479594.0,1.218612e+07,12.141637


In [4]:
# 3. 5개 분모 대안별 비율
ratio_sensitivity = response[KEY + ["당해계획예산_백만원"]].merge(
    denominator_sensitivity,
    on=KEY,
    how="left",
    validate="one_to_many",
    indicator=True,
)
ratio_sensitivity["계획예산_총세출비율_pct"] = (
    ratio_sensitivity["당해계획예산_백만원"] / ratio_sensitivity["분모금액_백만원"] * 100
)
ratio_sensitivity["100pct_초과"] = ratio_sensitivity["계획예산_총세출비율_pct"].gt(100)

variant_summary = ratio_sensitivity.groupby("분모대안").agg(
    행=("연도", "size"),
    중앙값_pct=("계획예산_총세출비율_pct", "median"),
    최댓값_pct=("계획예산_총세출비율_pct", "max"),
    초과100pct=("100pct_초과", "sum"),
)
display(variant_summary)

,행,중앙값_pct,최댓값_pct,초과100pct
분모대안,,,,
광역본청_3개회계_순계,153,56.004873,131.024836,23
지역통합_3개회계_순계,153,15.945094,68.460163,0
지역통합_3개회계_총계,153,11.898455,49.045527,0
지역통합_4개회계_총계,153,10.839991,46.541816,0
지역통합_일반회계_순계,153,19.198058,83.628253,0


In [5]:
# 4. 결합·값 QA와 저장
main_qa = {
    "행": len(combined),
    "조인누락": int(combined["_merge"].ne("both").sum()),
    "키중복": int(combined.duplicated(KEY).sum()),
    "분모결측": int(combined["세출예산순계액_백만원"].isna().sum()),
    "분모0이하": int(combined["세출예산순계액_백만원"].le(0).sum()),
    "비율비유한": int((~np.isfinite(combined["계획예산_총세출비율_pct"])).sum()),
}
assert main_qa == {
    "행": 153,
    "조인누락": 0,
    "키중복": 0,
    "분모결측": 0,
    "분모0이하": 0,
    "비율비유한": 0,
}
assert len(ratio_sensitivity) == 765
assert ratio_sensitivity["분모대안"].nunique() == 5
assert ratio_sensitivity["_merge"].eq("both").all()
assert ratio_sensitivity.duplicated(["분모대안", *KEY]).sum() == 0
assert np.isfinite(ratio_sensitivity["계획예산_총세출비율_pct"]).all()

combined = combined.drop(columns="_merge").sort_values(KEY).reset_index(drop=True)
ratio_sensitivity = (
    ratio_sensitivity.drop(columns="_merge").sort_values(["분모대안", *KEY]).reset_index(drop=True)
)
combined.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
ratio_sensitivity.to_csv(SENSITIVITY_OUTPUT_PATH, index=False, encoding="utf-8-sig")
pd.testing.assert_frame_equal(
    pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig"), combined, check_dtype=False
)
pd.testing.assert_frame_equal(
    pd.read_csv(SENSITIVITY_OUTPUT_PATH, encoding="utf-8-sig"),
    ratio_sensitivity,
    check_dtype=False,
)
display(pd.DataFrame([main_qa]))

,행,조인누락,키중복,분모결측,분모0이하,비율비유한
0,153,0,0,0,0,0


In [6]:
# 5. 주 비율 분포와 범위 경고
main_summary = combined["계획예산_총세출비율_pct"].describe(percentiles=[0.25, 0.5, 0.75])
scope_warning = ratio_sensitivity.loc[
    ratio_sensitivity["100pct_초과"],
    ["지역", "연도", "분모대안", "계획예산_총세출비율_pct"],
]
display(main_summary.to_frame("주_비율_pct"))
display(scope_warning.head(10))
print("saved:", OUTPUT_PATH.relative_to(ROOT), "/", len(combined), "rows")
print("saved:", SENSITIVITY_OUTPUT_PATH.relative_to(ROOT), "/", len(ratio_sensitivity), "rows")

,주_비율_pct
count,153.000000
mean,17.443910
std,8.365969
min,3.249481
25%,12.700896
50%,15.945094
75%,19.562465
max,68.460163


,지역,연도,분모대안,계획예산_총세출비율_pct
7,강원,2023,광역본청_3개회계_순계,105.504632
8,강원,2024,광역본청_3개회계_순계,115.161169
13,경기,2020,광역본청_3개회계_순계,106.433958
14,경기,2021,광역본청_3개회계_순계,102.566848
16,경기,2023,광역본청_3개회계_순계,113.864402
17,경기,2024,광역본청_3개회계_순계,117.135540
21,경남,2019,광역본청_3개회계_순계,104.885655
23,경남,2021,광역본청_3개회계_순계,102.911858
24,경남,2022,광역본청_3개회계_순계,103.781940
25,경남,2023,광역본청_3개회계_순계,109.558308


saved: data/processed/analysis/2016-2024_재정대응지수_총세출결합_패널.csv / 153 rows
saved: data/processed/analysis/2016-2024_재정대응지수_총세출_민감도_패널.csv / 765 rows


## Takeaways

- 주 비율 153건과 5개 대안별 비율 765건의 결합 QA를 통과했다.
- 주 비율은 명목 계획예산을 지역통합 3개 회계 당초예산 순계로 나눈 값이다.
- 광역본청 대안의 100% 초과는 행정범위 불일치 가능성을 보여 주므로 주 지표로 쓰지 않는다.